# Практика § 5.5: память и цена округления

Расчёты памяти дают нижние ориентиры для весов; фактический расход проверяйте на устройстве. Код не упаковывает 4-битные числа.

## Параметры и байты

Добавьте свой размер скрытого слоя и сравните форматы хранения.

In [ ]:
def parameters(a,b,c):
    return a*b+b+b*c+c
for dims in [(100,64,3),(100,128,3),(1000,64,10)]:
    n=parameters(*dims)
    print(dims,n,'fp16 bytes',n*2,'int8 bytes',n)

## Сжатие одного набора весов

Измените масштаб и сравните среднюю ошибку восстановления.

In [ ]:
import numpy as np
weights=np.array([.1,.4,.9])
for scale in [.1,.25,.5]:
    code=np.clip(np.rint(weights/scale),-8,7).astype(np.int8)
    restored=code.astype(float)*scale
    print(scale,code,restored,'MAE',round(np.mean(np.abs(weights-restored)),3))

## Изменение класса

Найдите вход, для которого порядок двух оценок поменяется после округления. Можно менять первый набор весов.

In [ ]:
w1=np.array([.1,.4,.9])
w2=np.array([.2,.3,.8])
input_vector=np.array([1.,1.,1.])
def quantize(w,scale=.25):
    return np.clip(np.rint(w/scale),-8,7)*scale
print('до',w1@input_vector,w2@input_vector)
print('после',quantize(w1)@input_vector,quantize(w2)@input_vector)

## Самостоятельно

Для 7 млрд параметров оцените только веса при 16, 8 и 4 битах. Затем назовите три дополнительные статьи памяти при обучении.

## Самопроверка




### 1. Сколько параметров у линейного слоя 10→5 со смещениями?

A. 15

B. 50

C. 55

D. 60




### 2. Сколько байт занимают 6659 весов по 2 байта?

A. 3329

B. 6659

C. 13318

D. 26636




### 3. Сколько памяти занимают только 7 млрд весов по 2 байта?

A. 3,5 ГБ

B. 7 ГБ

C. 14 ГБ

D. 112 ГБ




### 4. Почему 16 ГБ видеопамяти не гарантируют запуск модели с 14 ГБ весов?

A. Нужны также входы, промежуточные и служебные данные

B. Потому что 14 больше 16

C. Потому что веса не используются

D. Потому что все данные должны быть на диске




### 5. Что добавляется при обучении по сравнению с простым запуском?

A. Градиенты и состояние оптимизатора

B. Только новые JPEG

C. Только подписи классов

D. Ничего




### 6. Должны ли все 20 ГБ файлов снимков быть одновременно в видеопамяти?

A. Да, всегда

B. Нет, их можно читать пакетами

C. Только если модель маленькая

D. Только при float16




### 7. Что делает простая квантизация веса?

A. Округляет его до дискретного уровня с масштабом

B. Учит нового ученика

C. Меняет правильную метку

D. Удаляет весь слой




### 8. Во что восстановится 0,4 при масштабе 0,25 и обычном округлении?

A. 0,25

B. 0,4

C. 0,5

D. 1




### 9. Упаковывает ли NumPy int8 автоматически два 4-битных кода в байт?

A. Да

B. Нет

C. Только на GPU

D. Только для отрицательных




### 10. Что нужно проверить после квантизации помимо размера файла?

A. Качество ответов и фактическую скорость на нужном устройстве

B. Только число эпох учителя

C. Только цвет SVG

D. Только размер исходного текста




## Подсказки и решения

### Вопрос 1

**Подсказка**


Нужны веса для каждой пары вход–выход и пять смещений.



**Решение**


**C.** 10·5+5=55.



### Вопрос 2

**Подсказка**


Умножьте число параметров на размер одного.



**Решение**


**C.** 6659·2=13318 байт.



### Вопрос 3

**Подсказка**


Умножьте число параметров на 2 байта.



**Решение**


**C.** 7·10^9·2=14·10^9 байт, или 14 десятичных ГБ.



### Вопрос 4

**Подсказка**


Что хранится кроме весов?



**Решение**


**A.** Для вычисления нужны дополнительные тензоры и место работы библиотеки.



### Вопрос 5

**Подсказка**


Вспомните обратный проход.



**Решение**


**A.** Градиенты, состояния оптимизатора и активации увеличивают расход памяти.



### Вопрос 6

**Подсказка**


Как DataLoader подаёт данные?



**Решение**


**B.** Обычно файлы хранят на диске и подают пакетами.



### Вопрос 7

**Подсказка**


Посмотрите на q=round(w/s).



**Решение**


**A.** Вес заменяется кодом из ограниченного набора уровней.



### Вопрос 8

**Подсказка**


0,4/0,25=1,6 округляется до 2.



**Решение**


**C.** Код 2 восстанавливается как 2·0,25=0,5.



### Вопрос 9

**Подсказка**


Название типа сообщает размер одного элемента.



**Решение**


**B.** int8 хранит каждый код в одном байте; упаковка требует отдельной реализации.



### Вопрос 10

**Подсказка**


Сжатие может изменить числа и стоимость операций.



**Решение**


**A.** Решение принимают по памяти, скорости и качеству на своей задаче.



